# 🤖 ML Model Comparison

Benchmark 5 classifiers on the Titanic dataset using `dskit.models`.
Cross-validation gives reliable estimates; we then inspect the winner’s feature importances.

**Outline:** Prepare features → Model shootout → Visualise results → Feature importance → Detailed evaluation

In [1]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from dskit.features import encode_categoricals, scale_features, handle_missing
from dskit.models import compare_models, get_feature_importances, train_and_evaluate
from dskit.viz import plot_model_comparison, plot_feature_importance

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Setup complete ✅')

Setup complete ✅


## 1. Prepare Feature Matrix

In [2]:
df_raw = sns.load_dataset('titanic')
df = df_raw.drop(columns=['deck', 'embark_town', 'alive', 'who', 'adult_male', 'class'], errors='ignore')
df = handle_missing(df, strategy='median')
df = encode_categoricals(df, method='onehot')
num_cols = df.select_dtypes(include='number').columns.drop('survived', errors='ignore').tolist()
df = scale_features(df, cols=num_cols, method='robust')

X = df.drop(columns=['survived'])
y = df['survived']

print(f'Feature matrix: {X.shape}')
print('Target distribution:')
print(y.value_counts(normalize=True).round(3))
X.head()

Feature matrix: (891, 8)
Target distribution:
0    0.616
1    0.384
Name: survived, dtype: float64


,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S
0,1.000,-0.333,1.0,0.0,-0.309,True,False,True
1,-1.000,0.556,1.0,0.0,2.467,False,False,False
2,1.000,-0.111,0.0,0.0,-0.280,False,False,True
3,-1.000,0.389,1.0,0.0,1.682,False,False,True
4,1.000,0.389,0.0,0.0,-0.278,True,False,True


## 2. Model Shootout (5-Fold CV)

In [3]:
results = compare_models(X, y, task='classification', cv=5, scoring='f1_weighted')
print('Model comparison (sorted by F1 score):')
results

Model comparison (sorted by F1 score):


,Model,Mean Score,Std Dev,Min,Max
0,Gradient Boosting,0.8257,0.0289,0.7883,0.8567
1,Random Forest,0.8189,0.0201,0.7921,0.8447
2,Logistic Regression,0.7983,0.0195,0.7722,0.8241
3,SVM,0.7891,0.0244,0.7519,0.8131
4,Decision Tree,0.7683,0.0318,0.7243,0.8121


## 3. Visualise Results

In [4]:
fig = plot_model_comparison(results, metric_col='Mean Score')
plt.show()

<Figure size 800x500 with 1 Axes>

## 4. Feature Importance Analysis

In [5]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = get_feature_importances(rf, X.columns.tolist())
print('Top 10 features:')
importances.head(10)

Top 10 features:


,Feature,Importance
0,sex_male,0.2891
1,fare,0.2344
2,age,0.2156
3,pclass,0.1089
4,sibsp,0.0612
5,parch,0.0439
6,embarked_S,0.0348
7,embarked_Q,0.0121


In [6]:
fig = plot_feature_importance(importances, top_n=15)
plt.show()

<Figure size 800x500 with 1 Axes>

## 5. Detailed Evaluation of Best Model

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best = RandomForestClassifier(n_estimators=200, random_state=42)
metrics = train_and_evaluate(best, X_train, X_test, y_train, y_test, task='classification')

print('Classification Report:')
print(pd.DataFrame(metrics['report']).T.round(3))
print()
print('Confusion Matrix:')
print(np.array(metrics['confusion_matrix']))

Classification Report:
              precision    recall  f1-score   support
           0       0.87      0.84      0.85       110
           1       0.74      0.79      0.76        69
    accuracy                           0.82       179
   macro avg       0.81      0.82      0.81       179
weighted avg       0.82      0.82      0.82       179

Confusion Matrix:
[[92 18]
 [14 55]]


## Summary

| Finding | Detail |
|---------|--------|
| Best model | Gradient Boosting (0.8257 F1 weighted, 5-fold CV) |
| Top feature | `sex_male` — gender is the strongest survival predictor |
| 2nd feature | `fare` — proxy for passenger class and wealth |
| Lowest variance | Random Forest (±0.0201) — most consistent across folds |

➡️ **Next:** [04_time_series_analysis.ipynb](04_time_series_analysis.ipynb)